In [1]:
import gc, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from google.colab import drive, runtime

IN_COLAB = True
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Colab Notebooks/IV Project-Apr 3, 2026')
sys.path.insert(0, str(ROOT))

from src.paths import CLEAN_DATA_V2, OUTPUT
from src.benchmark import analytic_benchmark
from src.helper import _notebook_stem
from src.gru import GRUModel
from src.model3_utils import (
    FEATURE_SETS, TARGET, LOOKBACK,
    assign_sequences_to_splits, scale_sequences,
    SequenceDataset, detect_device, compute_batch_size,
    train_seq_model, predict_seq,
    hw_predict_aligned, save_seq_run,
    print_config, print_feature_set_summary,
)

Mounted at /content/drive


## Configuration

In [2]:
DATA_SET = 'chro_A'
NOTEBOOK_NAME = _notebook_stem()

SEED = 42
MAX_EPOCHS = 100
PATIENCE = 25
LR_PATIENCE = 8
LR_FACTOR = 0.3
WARMUP_EPOCHS = 5
HIDDEN_SIZE = 64
NUM_LAYERS = 2
DROPOUT = 0.1
BASE_LR = 1e-3
BASE_BATCH = 512

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)

## Load data

In [3]:
df_train = pd.read_parquet(CLEAN_DATA_V2 / f'{DATA_SET}_train_v2.parquet')
df_val   = pd.read_parquet(CLEAN_DATA_V2 / f'{DATA_SET}_val_v2.parquet')
df_test  = pd.read_parquet(CLEAN_DATA_V2 / f'{DATA_SET}_test_v2.parquet')

df_full = pd.concat([df_train, df_val, df_test], ignore_index=True)

print(f'Train: {len(df_train):,}  Val: {len(df_val):,}  Test: {len(df_test):,}')
print(f'Full data for sequences: {len(df_full):,} rows')

Train: 2,266,676  Val: 942,247  Test: 579,718
Full data for sequences: 3,788,641 rows


## GPU auto-detect

In [4]:
CFG = detect_device()
DEVICE = CFG['DEVICE']
AMP_DTYPE = CFG['POLICY']
USE_AMP = (AMP_DTYPE != torch.float32)

L4  |  VRAM: 24 GB  |  MAX_BATCH=2,048  |  dtype=torch.bfloat16


## Analytic benchmark

In [5]:
hw = analytic_benchmark(df_train, df_val, df_test, target=TARGET)
hw_coef = hw['coef']

Analytic Benchmark
SSE = 22.8915  RMSE = 0.006284
Coefficients: a = -0.142840, b = -0.082050, c = -0.058247


## Train across feature sets

Edit `FEATURE_SETS` in `src/model3_utils.py` to add/remove sets,
or override locally:
```python
from src.model3_utils import FEATURE_SETS
FEATURE_SETS = {k: v for k, v in FEATURE_SETS.items() if k in ('4F', '8F')}
```

In [6]:
OUTPUT_DIR = OUTPUT / NOTEBOOK_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

results_by_fs = {}
df_sorted_ref = None

pbar = tqdm(FEATURE_SETS.items(), desc='Feature sets', unit='set')
for fs_name, feature_cols in pbar:
    pbar.set_postfix_str(fs_name)
    print_feature_set_summary(fs_name, 0, 0, 0, feature_cols)

    # Build sequences and assign to splits
    seq_data = assign_sequences_to_splits(
        df_full, df_train, df_val, df_test,
        feature_cols=feature_cols, target=TARGET, lookback=LOOKBACK,
    )
    X_tr, y_tr = seq_data['X_train'], seq_data['y_train']
    X_va, y_va = seq_data['X_val'], seq_data['y_val']
    X_te, y_te = seq_data['X_test'], seq_data['y_test']
    test_idx = seq_data['test_indices']
    df_sorted = seq_data['df_sorted']
    if df_sorted_ref is None:
        df_sorted_ref = df_sorted

    print(f'  Sequences — Train: {len(X_tr):,}  Val: {len(X_va):,}  Test: {len(X_te):,}')

    # Scale
    X_tr_sc, X_va_sc, X_te_sc, scaler = scale_sequences(X_tr, X_va, X_te)

    # DataLoaders
    BATCH_SIZE = compute_batch_size(len(X_tr), CFG['MAX_BATCH'])
    INIT_LR = BASE_LR * (BATCH_SIZE / BASE_BATCH) ** 0.5
    print_config(CFG, BATCH_SIZE, INIT_LR, len(X_tr), MAX_EPOCHS, PATIENCE, WARMUP_EPOCHS, LOOKBACK)

    train_loader = DataLoader(SequenceDataset(X_tr_sc, y_tr), batch_size=BATCH_SIZE,
                              shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
    val_loader = DataLoader(SequenceDataset(X_va_sc, y_va), batch_size=BATCH_SIZE,
                            shuffle=False, num_workers=2, pin_memory=True)

    # Model
    model = GRUModel(
        n_features=len(feature_cols), hidden_size=HIDDEN_SIZE,
        num_layers=NUM_LAYERS, dropout=DROPOUT, seed=SEED,
    ).to(DEVICE)
    print(f'  GRU params: {sum(p.numel() for p in model.parameters()):,}')

    # Train
    train_result = train_seq_model(
        model, train_loader, val_loader,
        device=DEVICE, amp_dtype=AMP_DTYPE, use_amp=USE_AMP,
        max_epochs=MAX_EPOCHS, patience=PATIENCE,
        lr_patience=LR_PATIENCE, lr_factor=LR_FACTOR,
        init_lr=INIT_LR, warmup_epochs=WARMUP_EPOCHS,
    )

    # Predict & evaluate
    y_pred = predict_seq(train_result['model'], X_te_sc, DEVICE, AMP_DTYPE, USE_AMP)
    _, _, hw_sse = hw_predict_aligned(hw_coef, df_sorted, test_idx)

    from src.metrics import metrics, gain
    met = metrics(y_te, y_pred)
    g = gain(met['SSE'], hw_sse) * 100
    print(f'  {fs_name}: SSE={met["SSE"]:.4f}  RMSE={met["RMSE"]:.6f}  '
          f'Gain={g:+.2f}%  epochs={train_result["epochs"]}  '
          f'time={train_result["training_time"]:.1f}s')

    results_by_fs[fs_name] = {
        'model': train_result['model'],
        'y_pred': y_pred, 'y_true': y_te,
        'test_indices': test_idx,
        'history': train_result['history'],
        'epochs': train_result['epochs'],
        'training_time': train_result['training_time'],
        'scaler': scaler,
    }

    # Free memory
    del X_tr, X_va, X_te, X_tr_sc, X_va_sc, X_te_sc, train_loader, val_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

Feature sets:   0%|          | 0/7 [00:00<?, ?set/s]


  Feature set: 3F  (3 features)
  Train: 0  Val: 0  Test: 0 sequences
  Features: delta, T, spy_ret
  Sequences — Train: 1,469,381  Val: 648,019  Test: 426,055
MAX_BATCH=2,048  adaptive BATCH_SIZE=2,048  INIT_LR=0.002000  n_train=1,469,381  steps/epoch~717
MAX_EPOCHS=100  PATIENCE=25  WARMUP=5 epochs  LOOKBACK=20
  GRU params: 38,273
  3F: SSE=21.5035  RMSE=0.007104  Gain=-52.81%  epochs=32  time=297.5s

  Feature set: 5F  (5 features)
  Train: 0  Val: 0  Test: 0 sequences
  Features: delta, T, spy_ret, vix_lag, vix_mom_lag
  Sequences — Train: 1,469,381  Val: 648,019  Test: 426,055
MAX_BATCH=2,048  adaptive BATCH_SIZE=2,048  INIT_LR=0.002000  n_train=1,469,381  steps/epoch~717
MAX_EPOCHS=100  PATIENCE=25  WARMUP=5 epochs  LOOKBACK=20
  GRU params: 38,657
  5F: SSE=23.7182  RMSE=0.007461  Gain=-68.55%  epochs=38  time=292.1s

  Feature set: 6F_G  (6 features)
  Train: 0  Val: 0  Test: 0 sequences
  Features: delta, T, spy_ret, vix_lag, vix_mom_lag, gamma
  Sequences — Train: 1,469,381

## Save results

In [7]:
summary = save_seq_run(
    OUTPUT_DIR,
    results_by_fs=results_by_fs,
    hw_coef=hw_coef,
    df_sorted=df_sorted_ref,
)
print('\nMetrics Summary:')
summary

Saving results to: /content/drive/MyDrive/Colab Notebooks/IV Project-Apr 3, 2026/output/4.0-gru-chro-A-colab
Feature sets to save: ['3F', '5F', '6F_G', '6F_R', '6F_T', '8F_GT', '8F_GR']

✓ Saved metrics_summary.csv
✓ Saved residual_diagnostics.csv
✓ Saved gain_table.csv
✓ Saved 7 × 3 artifacts (weights, predictions, history)
✓ All results saved to /content/drive/MyDrive/Colab Notebooks/IV Project-Apr 3, 2026/output/4.0-gru-chro-A-colab

Metrics Summary:


,Model,SSE,MSE,RMSE,MAE,MeanError,MedianAE,R2,Training_time,Gain_vs_Analytic
0,Analytic (3F),14.071868,0.000033,0.005747,0.003191,0.000598,0.001905,0.183376,None,None
1,3F,21.503500,0.000050,0.007104,0.003728,0.000095,0.002306,-0.247900,297.5s,-52.81%
2,Analytic (5F),14.071868,0.000033,0.005747,0.003191,0.000598,0.001905,0.183376,None,None
3,5F,23.718182,0.000056,0.007461,0.003480,0.000225,0.001957,-0.376423,292.1s,-68.55%
4,Analytic (6F_G),14.071868,0.000033,0.005747,0.003191,0.000598,0.001905,0.183376,None,None
5,6F_G,19.291330,0.000045,0.006729,0.002953,-0.000683,0.001640,-0.119523,299.0s,-37.09%
6,Analytic (6F_R),14.071868,0.000033,0.005747,0.003191,0.000598,0.001905,0.183376,None,None
7,6F_R,21.160067,0.000050,0.007047,0.003495,-0.000225,0.002085,-0.227970,243.6s,-50.37%
8,Analytic (6F_T),14.071868,0.000033,0.005747,0.003191,0.000598,0.001905,0.183376,None,None
9,6F_T,22.255142,0.000052,0.007227,0.003495,-0.000316,0.001995,-0.291520,241.6s,-58.15%


## Summary

In [8]:
total_time = sum(r['training_time'] for r in results_by_fs.values())
print(f'\nGRU on {DATA_SET} — total training time: {total_time / 60:.1f} min')
for fs_name, res in results_by_fs.items():
    from src.metrics import metrics, gain
    met = metrics(res['y_true'], res['y_pred'])
    _, _, hw_sse = hw_predict_aligned(hw_coef, df_sorted_ref, res['test_indices'])
    g = gain(met['SSE'], hw_sse) * 100
    print(f'  {fs_name}: SSE={met["SSE"]:.4f}  Gain={g:+.2f}%  epochs={res["epochs"]}')


GRU on chro_A — total training time: 36.5 min
  3F: SSE=21.5035  Gain=-52.81%  epochs=32
  5F: SSE=23.7182  Gain=-68.55%  epochs=38
  6F_G: SSE=19.2913  Gain=-37.09%  epochs=38
  6F_R: SSE=21.1601  Gain=-50.37%  epochs=31
  6F_T: SSE=22.2551  Gain=-58.15%  epochs=31
  8F_GT: SSE=15.3745  Gain=-9.26%  epochs=47
  8F_GR: SSE=12.6590  Gain=+10.04%  epochs=54


## Cleanup

In [9]:
del results_by_fs, df_full, df_sorted_ref
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f'Final VRAM: {(total - free) / 1e9:.2f} GB / {total / 1e9:.0f} GB')

print(f'Total training time: {total_time / 60:.1f} min')

if IN_COLAB:
    runtime.unassign()

Final VRAM: 0.40 GB / 24 GB
Total training time: 36.5 min
